# Core EDA

In [1]:
import sys
sys.path.insert(0, '/home/lu72hip/DIGICHer/dh_pipeline/src')

import pandas as pd
from pathlib import Path
from lib.database.duck.create_connection import create_duck_connection
from utils.config.config_loader import get_query_config

config = get_query_config()["core_v3"]
# DB = Path(config['path_staging_duck'])
# DB = Path(config['path_topics_duck'])
DB = Path(config['path_final_duck'])

con = create_duck_connection(str(DB))
con.execute("SET memory_limit='128GB'")
con.execute("SET threads=8")
print(f'Connected to {DB}')

pd.set_option("display.max_rows", 50) # Show more rows  # or None for unlimited
pd.set_option("display.max_columns", None) # Show more columns  # None = show all
pd.set_option("display.max_colwidth", None) # Don't truncate column content  # None = full content
pd.set_option("display.width", None) # Wider display  # Auto-detect terminal width

Connected to /vast/lu72hip/data/duckdb/core/core_v3_final.duckdb


## Stats

In [2]:
q = """ 
SELECT
    table_name,
    estimated_size as cnt
FROM duckdb_tables()
WHERE schema_name = 'main'
ORDER BY estimated_size DESC
"""
con.execute(q).df()

,table_name,cnt
0,relation,562206510
1,work,205841448
2,project,3656773
3,relation_topic,3648111
4,organization,448161
5,topic,4516


## Projects and their Topics

In [3]:
q = """ 
SELECT *
FROM relation_topic
LIMIT 3
"""
con.execute(q).df()

,type,source_id,topic_id,score,created_at
0,project,3722803268331,14316,0.170100,NaT
1,project,5977149805285,11295,0.226497,NaT
2,project,196473757163,14394,0.274071,NaT


In [6]:
q = """ 
SELECT p.title, t.topic_name
FROM project p 
LEFT JOIN relation_topic rt on rt.source_id = p.id
LEFT JOIN topic t on rt.topic_id = t.id
LIMIT 3
"""
con.execute(q).df()

,title,topic_name
0,Natural language processing for precision medicine and clinical and consumer health question,Mental Health via Writing
1,MOLECULAR GENETICS OF INHERITED NEUROLOGIC DISEASES,Gestational Trophoblastic Disease Studies
2,The Effect of Saturated Fat Ingestion on the HDL Proteome,Paraoxonase enzyme and polymorphisms


## Cordis and ROR enrichment

In [7]:
q = """ 
SELECT count(*)
FROM organization
where geolocation is not NULL
LIMIT 3
"""
con.execute(q).df()

,count_star()
0,123541


In [8]:
q = """ 
SELECT count(*)
FROM relation
where cordis_ec_contribution is not NULL
LIMIT 3
"""
con.execute(q).df()

,count_star()
0,188975


In [9]:
# Top 5 institutions by total EC funding received — CNRS should be #1
q = """
SELECT
    o.legalName,
    SUM(r.cordis_ec_contribution)   AS total_ec_contribution,
    COUNT(*)                         AS project_count
FROM relation r
JOIN organization o ON r.target = o.id
WHERE r.sourceType = 'project'
  AND r.targetType = 'organization'
  AND r.cordis_ec_contribution IS NOT NULL
GROUP BY o.legalName
ORDER BY total_ec_contribution DESC
LIMIT 10
"""
con.execute(q).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,legalName,total_ec_contribution,project_count
0,CENTRE NATIONAL DE LA RECHERCHE SCIENTIFIQUE CNRS,2.055781e+09,2595
1,COMMISSARIAT A L ENERGIE ATOMIQUE ET AUX ENERGIES ALTERNATIVES,1.684705e+09,1873
2,"THE CHANCELLOR, MASTERS AND SCHOLARS OF THE UNIVERSITY OF OXFORD",1.072649e+09,1548
3,University College London,8.250836e+08,1352
4,INSTITUT NATIONAL DE LA SANTE ET DE LA RECHERCHE MEDICALE,7.984103e+08,1000
5,IMPERIAL COLLEGE OF SCIENCE TECHNOLOGY AND MEDICINE,6.944187e+08,1303
6,KOBENHAVNS UNIVERSITET,5.595072e+08,1091
7,Weizmann Institute of Science,5.573708e+08,551
8,Teknologian tutkimuskeskus VTT Oy,5.334034e+08,855
9,COST ASSOCIATION,5.330092e+08,4


# CH Classification

In [4]:
# ...
q = """
SELECT count(*)
FROM project
WHERE pred < 0.1
LIMIT 10
"""
con.execute(q).df()

,count_star()
0,1804995
